____
# Add editing flag and distance to coast

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import dask.dataframe as dd

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.scheduler.transition-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.comm.recent-messages-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(


In [2]:
p = 15
if True:
    from dask.distributed import Client
    from dask_jobqueue import PBSCluster

    # cluster = PBSCluster(cores=56, processes=28, walltime='04:00:00')
    # cluster = PBSCluster(cores=7, processes=7, walltime='04:00:00')
    cluster = PBSCluster(cores=p, processes=p, walltime="04:00:00")
    w = cluster.scale(jobs=4)
else:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()

client = Client(cluster)
client

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://10.148.1.62:8787/status,
Dashboard: http://10.148.1.62:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.148.1.62:38140,Workers: 0
Dashboard: http://10.148.1.62:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [13]:
cluster.close()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/

_______
# Data

In [9]:
df = (pd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')), dtype={'drifter_id': 'str', 'pass_number':'int'})
     ).set_index('row_number')[['longitude', 'latitude', 'pass_number', 'cycle_number']]
pass_number = df.pass_number.unique()
for swath in pass_number : 
    cycle_number = df.where(df.pass_number==swath).dropna().cycle_number.unique()
    for cycle in cycle_number:
        df.where(df.pass_number==swath).where(df.cycle_number==cycle).dropna().to_csv(os.path.join(zarr_dir, 'coloc_pass_cycle', f'{int(swath)}_{int(cycle)}.csv'))
        print(swath, cycle)
                                                                                      

3 478.0
3 479.0
3 480.0
3 481.0
3 482.0
3 483.0
3 484.0
3 485.0
3 486.0
3 487.0
3 488.0
3 489.0
3 490.0
3 491.0
3 492.0
3 493.0
3 494.0
3 495.0
3 496.0
3 497.0
3 499.0
3 500.0
3 501.0
3 502.0
3 503.0
3 504.0
3 505.0
3 506.0
3 507.0
3 508.0
3 509.0
3 510.0
3 511.0
3 512.0
3 513.0
3 514.0
3 515.0
3 516.0
3 517.0
3 518.0
3 519.0
3 520.0
3 521.0
3 522.0
3 523.0
3 524.0
3 525.0
3 529.0
3 530.0
3 531.0
3 532.0
3 533.0
3 534.0
3 535.0
3 536.0
3 537.0
3 538.0
3 539.0
3 540.0
3 541.0
3 542.0
3 543.0
3 544.0
3 545.0
3 546.0
3 547.0
3 548.0
3 549.0
3 550.0
3 551.0
3 552.0
3 553.0
3 554.0
3 555.0
3 556.0
3 557.0
3 558.0
3 559.0
3 560.0
3 561.0
3 562.0
3 563.0
3 564.0
3 565.0
3 567.0
3 569.0
3 570.0
3 571.0
3 573.0
3 574.0
3 575.0
3 576.0
3 577.0
3 578.0
16 502.0
16 503.0
16 504.0
16 505.0
16 506.0
16 507.0
16 509.0
16 510.0
16 511.0
16 512.0
16 514.0
16 515.0
16 516.0
16 517.0
16 518.0
16 519.0
16 520.0
16 521.0
16 522.0
16 523.0
16 524.0
16 525.0
16 529.0
16 530.0
16 531.0
16 532.0
16 533.0
16 53

In [3]:
dfs = browse_swot_250().reset_index()
drifters_sources = 'all_med_variational_10min_v0.nc'
#drifters_sources = 'all_med_lowess_10min_v0.nc'
#df = (dd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')), dtype={'drifter_id': 'str', 'pass_number':'int'})
#     ).set_index('row_number')[['longitude', 'latitude', 'pass_number', 'cycle_number']]

files = sorted(glob(os.path.join(zarr_dir, 'coloc_pass_cycle', '*.csv')))
files

['/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_502.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_503.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_504.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_505.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_506.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_507.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_509.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_510.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_511.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_512.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_514.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_515.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/col

__________
# Editing_flag

In [4]:
cycle = 478
swath = 3

def get_editing_flag(lon, lat, dss):
    R= 6378e3
    diag = 250# * np.sqrt(2)
    dlat = diag*360/(2*np.pi*R)
    dlon = diag*360/(2*np.pi*R*np.cos(lat*np.pi/180))
    testlon = (dss.longitude > lon-dlon) & (dss.longitude < lon+dlon)
    testlat = (dss.latitude > lat-dlat) & (dss.latitude < lat+dlat)
    l = dss.where(testlon &testlat, drop=True).duacs_editing_flag.fillna(int(200)).values
    return '_'.join(l.reshape((1, np.size(l))).astype(int).astype(str)[0])

def apply_get_editing_flag_one(df_, dss):
    l = get_editing_flag(df_.longitude, df_.latitude, dss)
    return l

def apply_get_editing_flag_group(df, pass_number, cycle_number):
    #pass_number = int(df.pass_number.mean())
    #cycle_number = int(df.cycle_number.mean())
    try : 
        dss = xr.open_dataset(dfs.where((dfs.pass_number==pass_number)&(dfs.cycle_number==cycle_number)).dropna().file.values[0])[['duacs_editing_flag']]
    except : 
        assert False, (pass_number, cycle_number)
    
    dfe = df.apply(apply_get_editing_flag_one, args=(dss,), axis=1)
    return dfe.rename('editing_flag')
    

l = apply_get_editing_flag_group(pd.read_csv(files[0]).set_index('row_number').iloc[0:100], 3, 478)

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


In [13]:
#ef = df.groupby(['cycle_number', 'pass_number']).apply(apply_get_editing_flag_group, meta=l)
#eff = ef.compute()

In [ ]:
def editing_flag_all():
    for f in files : 
        df_ = dd.read_csv(f).repartition(npartitions=50).persist()
        df_.map_partitions(apply_get_editing_flag_group, pass_number=swath, cycle_number=cycle, meta=l).to_csv(f.replace('coloc_pass_cycle', 'editing_flags'))
        print(f)
    
editing_flag_all()

/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_502.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_503.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_504.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_505.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_506.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_507.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_509.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_510.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_511.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_512.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_514.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_515.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_516.csv
/home/datawork-lops-oc/ap

In [5]:
def editing_flag_all(df):
    DF = []
    pass_number = df.pass_number.unique().compute()
    for swath in pass_number : 
        cycle_number = df.where(df.pass_number==swath).dropna().cycle_number.unique().compute()
        for cycle in cycle_number:
            df_ = df.where(df.pass_number==swath).where(df.cycle_number==cycle).dropna().repartition(npartitions=3*p).persist()
            df_.map_partitions(apply_get_editing_flag_group, pass_number=swath, cycle_number=cycle, meta=l).compute()#.to_parquet(os.path.join(zarr_dir, 'editing_flags', f'{swath}_{cycle}.parquet'))
            print(swath, cycle)
    
editing_flag_all(df)


KeyboardInterrupt

